## _Import Libraries_

In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## _Import Dataset_

In [2]:
df = pd.read_csv('../data/parkinson_data.csv')

In [3]:
df.head(10)

,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335
5,phon_R01_S01_6,120.552,131.162,113.787,0.00968,0.00008,0.00463,0.00750,0.01388,0.04701,...,0.06985,0.01222,21.378,1,0.415564,0.825069,-4.242867,0.299111,2.187560,0.357775
6,phon_R01_S02_1,120.267,137.244,114.820,0.00333,0.00003,0.00155,0.00202,0.00466,0.01608,...,0.02337,0.00607,24.886,1,0.596040,0.764112,-5.634322,0.257682,1.854785,0.211756
7,phon_R01_S02_2,107.332,113.840,104.315,0.00290,0.00003,0.00144,0.00182,0.00431,0.01567,...,0.02487,0.00344,26.892,1,0.637420,0.763262,-6.167603,0.183721,2.064693,0.163755
8,phon_R01_S02_3,95.730,132.068,91.754,0.00551,0.00006,0.00293,0.00332,0.00880,0.02093,...,0.03218,0.01070,21.812,1,0.615551,0.773587,-5.498678,0.327769,2.322511,0.231571
9,phon_R01_S02_4,95.056,120.103,91.226,0.00532,0.00006,0.00268,0.00332,0.00803,0.02838,...,0.04324,0.01022,21.862,1,0.547037,0.798463,-5.011879,0.325996,2.432792,0.271362


In [4]:
# number of rows and columns in the dataframe
print("Dataset shape:", df.shape)

Dataset shape: (195, 24)


In [5]:
print("\nMissing values:")
print(df.isnull().sum().sum())


Missing values:
0


| Column Name           | Description                                                                                                                                      |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ |
| **name**              | Identifier or patient name. Each record corresponds to a particular visit for a patient.                                                         |
| **MDVP\:Fo(Hz)**      | Average vocal fundamental frequency (F0) in Hertz (Hz), a measure of pitch. Changes may be linked to Parkinson’s symptoms.                       |
| **MDVP\:Fhi(Hz)**     | Maximum vocal fundamental frequency during the recording (Hz).                                                                                   |
| **MDVP\:Flo(Hz)**     | Minimum vocal fundamental frequency during the recording (Hz).                                                                                   |
| **MDVP\:Jitter(%)**   | Variation in fundamental frequency relative to its mean, expressed as a percentage; higher jitter may indicate vocal instability.                |
| **MDVP\:Jitter(Abs)** | Absolute jitter in seconds; shows small fluctuations in frequency over time.                                                                     |
| **MDVP\:RAP**         | Relative amplitude perturbation; similar to jitter but measured over a short window, reflects microvariations in frequency.                      |
| **MDVP\:PPQ**         | Five-point period perturbation quotient; another measure of frequency irregularity.                                                              |
| **Jitter\:DDP**       | Difference of differences of periods; sensitive to irregularities in vocal frequency.                                                            |
| **MDVP\:Shimmer**     | Measure of amplitude variation across cycles; shows instability in vocal loudness.                                                               |
| **MDVP\:Shimmer(dB)** | Same as above but expressed in decibels.                                                                                                         |
| **Shimmer\:APQ3**     | Three-point amplitude perturbation quotient; reflects amplitude changes over short periods.                                                      |
| **Shimmer\:APQ5**     | Five-point amplitude perturbation quotient; longer window amplitude variation measure.                                                           |
| **MDVP\:APQ**         | Average amplitude perturbation quotient across cycles; overall loudness instability.                                                             |
| **Shimmer\:DDA**      | Difference of differences of amplitudes; another amplitude irregularity measure.                                                                 |
| **NHR**               | Noise-to-Harmonics Ratio; indicates how much noise is present in the voice signal, with higher values often associated with pathological speech. |
| **HNR**               | Harmonics-to-Noise Ratio; inverse of NHR, showing clarity of voice; lower values may indicate vocal deterioration.                               |
| **status**            | Health status indicator: **1 = Parkinson’s disease**, **0 = healthy control**. This is your target variable in classification tasks.             |
| **RPDE**              | Recurrence Period Density Entropy; measures the periodicity of the signal, with abnormal vocal patterns in Parkinson’s.                          |
| **DFA**               | Detrended Fluctuation Analysis; measures self-similarity and complexity in voice signals.                                                        |
| **spread1**           | Non-linear measure of variability in the voice signal.                                                                                           |
| **spread2**           | Another non-linear measure capturing the spread or variation in the signal.                                                                      |
| **D2**                | Correlation dimension; indicates complexity and chaotic behavior in time series data.                                                            |
| **PPE**               | Pitch Period Entropy; quantifies irregularities and disorder in pitch variation, often abnormal in Parkinson’s patients.

In [6]:
print("\nDuplicate rows:")
print(df.duplicated().sum())


Duplicate rows:
0


In [7]:
print("\nTarget distribution:")
print(df['status'].value_counts())


Target distribution:
status
1    147
0     48
Name: count, dtype: int64


In [9]:
# distribution of target Variable
df['status'].value_counts()

status
1    147
0     48
Name: count, dtype: int64

In [8]:
print("\nData types:")
print(df.dtypes)


Data types:
name                    str
MDVP:Fo(Hz)         float64
MDVP:Fhi(Hz)        float64
MDVP:Flo(Hz)        float64
MDVP:Jitter(%)      float64
MDVP:Jitter(Abs)    float64
MDVP:RAP            float64
MDVP:PPQ            float64
Jitter:DDP          float64
MDVP:Shimmer        float64
MDVP:Shimmer(dB)    float64
Shimmer:APQ3        float64
Shimmer:APQ5        float64
MDVP:APQ            float64
Shimmer:DDA         float64
NHR                 float64
HNR                 float64
status                int64
RPDE                float64
DFA                 float64
spread1             float64
spread2             float64
D2                  float64
PPE                 float64
dtype: object


## Data Pre-Processing

## _Splitting Target and Feature_

In [9]:
X = df.drop(columns=['name', 'status'])
y = df['status']

In [10]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (195, 22)
Target shape: (195,)


In [11]:
print("\nFeature columns:")
print(X.columns.tolist())


Feature columns:
['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']


In [12]:
print("\nTarget distribution:")
print(y.value_counts())


Target distribution:
status
1    147
0     48
Name: count, dtype: int64


## Splitting the data to training data & Test data

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=2,
    stratify=y
)

In [14]:
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (156, 22)
Testing data shape: (39, 22)


In [15]:
print("\nTraining target distribution:")
print(y_train.value_counts())


Training target distribution:
status
1    118
0     38
Name: count, dtype: int64


In [16]:
print("\nTesting target distribution:")
print(y_test.value_counts())


Testing target distribution:
status
1    29
0    10
Name: count, dtype: int64


## Build and Compare the Models

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

## _Define Models_

In [18]:
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "SVM Linear": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    )
}

## _Train and evaluate each model_

In [19]:
results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results[name] = accuracy

    print("=" * 60)
    print(name)
    print("=" * 60)

    print("Accuracy:", accuracy)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Healthy", "Parkinson's"]
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

Logistic Regression
Accuracy: 0.7692307692307693

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.53      0.90      0.67        10
 Parkinson's       0.95      0.72      0.82        29

    accuracy                           0.77        39
   macro avg       0.74      0.81      0.75        39
weighted avg       0.85      0.77      0.78        39

Confusion Matrix:
[[ 9  1]
 [ 8 21]]

SVM Linear
Accuracy: 0.8205128205128205

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.60      0.90      0.72        10
 Parkinson's       0.96      0.79      0.87        29

    accuracy                           0.82        39
   macro avg       0.78      0.85      0.79        39
weighted avg       0.87      0.82      0.83        39

Confusion Matrix:
[[ 9  1]
 [ 6 23]]

SVM RBF
Accuracy: 0.7948717948717948

Classification Report:
              precision    recall  f1-score   support

     Healthy     

C:\Users\91983\OneDrive\Desktop\Prediction of Disease Outbrakes\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\91983\OneDrive\Desktop\Prediction of Disease Outbrakes\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Random Forest
Accuracy: 0.9487179487179487

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.90      0.90      0.90        10
 Parkinson's       0.97      0.97      0.97        29

    accuracy                           0.95        39
   macro avg       0.93      0.93      0.93        39
weighted avg       0.95      0.95      0.95        39

Confusion Matrix:
[[ 9  1]
 [ 1 28]]



## Train the Final Random Forest Model

In [20]:
from sklearn.ensemble import RandomForestClassifier
final_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

# Train the final model
final_model.fit(X_train, y_train)

print("Final Random Forest model trained successfully.")

Final Random Forest model trained successfully.


## Evaluate the Final Random Forest

## _Make predictions_

In [21]:
y_pred = final_model.predict(X_test)

## _Accuracy_

In [22]:
accuracy = accuracy_score(y_test, y_pred)

print("Final Model Accuracy:", accuracy)
print("Final Model Accuracy: {:.2%}".format(accuracy))

# Classification Report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Healthy", "Parkinson's"]
    )
)


Final Model Accuracy: 0.9487179487179487
Final Model Accuracy: 94.87%

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.90      0.90      0.90        10
 Parkinson's       0.97      0.97      0.97        29

    accuracy                           0.95        39
   macro avg       0.93      0.93      0.93        39
weighted avg       0.95      0.95      0.95        39



## _Confusion Matrix_

In [23]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[ 9  1]
 [ 1 28]]


## _Save the Parkinson's Model_

In [24]:
import pickle

model_path = "../models/Parkinsons_model.sav"

with open(model_path, "wb") as file:
    pickle.dump(final_model, file)

print("Parkinson's model saved successfully!")
print("Saved at:", model_path)

Parkinson's model saved successfully!
Saved at: ../models/Parkinsons_model.sav


## _Load the saved model_

In [26]:
with open("../models/Parkinsons_model.sav", "rb") as file:
    loaded_model = pickle.load(file)

print("Loaded model type:", type(loaded_model))

Loaded model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


## Test the Saved Model with a Sample Patient

## _Create a sample patient_

In [27]:
sample_patient = pd.DataFrame([{
    'MDVP:Fo(Hz)': 119.992,
    'MDVP:Fhi(Hz)': 157.302,
    'MDVP:Flo(Hz)': 74.997,
    'MDVP:Jitter(%)': 0.00784,
    'MDVP:Jitter(Abs)': 0.00007,
    'MDVP:RAP': 0.00370,
    'MDVP:PPQ': 0.00554,
    'Jitter:DDP': 0.01109,
    'MDVP:Shimmer': 0.04374,
    'MDVP:Shimmer(dB)': 0.426,
    'Shimmer:APQ3': 0.02182,
    'Shimmer:APQ5': 0.03130,
    'MDVP:APQ': 0.02971,
    'Shimmer:DDA': 0.06545,
    'NHR': 0.02211,
    'HNR': 21.033,
    'RPDE': 0.414783,
    'DFA': 0.815285,
    'spread1': -4.813031,
    'spread2': 0.266482,
    'D2': 2.301442,
    'PPE': 0.284654
}])

## _Predict_

In [28]:
prediction = loaded_model.predict(sample_patient)

## _Prediction probability_

In [29]:
probability = loaded_model.predict_proba(sample_patient)

print("Prediction:", prediction[0])

print("\nProbability:")
print("Healthy:", probability[0][0])
print("Parkinson's:", probability[0][1])

Prediction: 1

Probability:
Healthy: 0.0033333333333333335
Parkinson's: 0.9966666666666667


## _Display final result_

In [30]:
if prediction[0] == 1:
    print("\nResult: Parkinson's Positive")
else:
    print("\nResult: Healthy")


Result: Parkinson's Positive
